# Poster results: reproducing every number and chart on the poster

One notebook, read top to bottom, regenerates every figure and every headline number that appears
on `poster/poster.html`, plus two checks that were added after the poster's first draft (the
hourly recovery-timing EDA in §2, and §6b) and had never been formalised anywhere. It reads
already-saved artifacts wherever they exist. Nothing here re-fits a model or re-runs conformal
calibration, so it is cheap (~1 min) and needs no GPU.

Sections match the poster's own card numbers.

## 0. Setup

In [1]:
import sys
sys.path.insert(0, "..")   # notebooks/ is one level down, so the repo root goes on the path

import warnings
warnings.filterwarnings("ignore")

import json
import shutil

import numpy as np
import pandas as pd

from src import orders, recovery
from src.utils import config, plots, store_view   # store_view: picks and renders the one-store example in §6c
from src.utils.features import censoring_bucket
from src.utils.metrics import lost_sales_vs_waste, quantile_scores, pinball_by_quantile, scores_by_bucket

# Shorthand paths used throughout - everything this notebook reads or writes lives under these
# three folders, plus a copy of each chart written straight into the poster's own images folder.
REPORTS = config.REPORTS_DIR
PLOTS = config.PLOTS_DIR
POSTER_IMAGES = config.ROOT / "poster" / "images"


## 2. Data: the censoring rate, and when a stockout first begins

The 43.8% headline is read straight off the subset summary. The hourly EDA is new: it answers
"once a shelf empties, does it tend to come back the same day?" Training-period days only,
matching every other recovery figure.

Broken down further just below the chart: an outage already under way at opening behaves differently from one that starts mid-day - see "does an outage stock back the same day" for the split.


In [2]:
daily = recovery.load_daily("recovered")
hourly = recovery.hours(daily)          # explode the daily frame back out to one row per (day, hour)
lo, hi = config.ACTIVE_HOURS

tr = hourly[hourly.period == "training"]
active = tr[(tr.hour >= lo) & (tr.hour < hi)]   # only the hours the store is actually meant to be open

# For every stockout day, which hours were actually recorded as out of stock.
out_hours = (active[active.hour_stockout == 1]
             .groupby(["store_id", "product_id", "dt"])["hour"].apply(list))
first_hour = out_hours.apply(min)     # the hour each stockout day first went empty
# Did the shelf ever get restocked (an in-stock hour shows up again) after it first ran out?
recovers_same_day = out_hours.apply(
    lambda hrs: any(h > min(hrs) and h not in hrs for h in range(min(hrs), hi)))

hours = np.arange(lo, hi)
# % of stockout-days whose first empty hour was each given hour.
pct = first_hour.value_counts().reindex(hours, fill_value=0) / len(first_hour) * 100

timing = pd.DataFrame({"hour": hours, "pct_of_stockout_days_starting_here": pct.values})
timing.to_csv(REPORTS / "stockout_timing.csv", index=False)

already_open_pct = float(pct.values[0])
never_recovers_pct = float(100 - recovers_same_day.mean() * 100)
avg_hours_out = float(out_hours.apply(len).mean())

print(f"n stockout-days (training period): {len(out_hours):,}")
print(f"already empty at opening (06:00): {already_open_pct:.1f}%")
print(f"never restock before closing, once started: {never_recovers_pct:.1f}%")
print(f"avg hours affected per stockout day: {avg_hours_out:.2f} of {hi - lo}")

loading recovered subset from data/processed


loading hourly subset from data/processed


n stockout-days (training period): 156,969
already empty at opening (06:00): 13.4%
never restock before closing, once started: 87.4%
avg hours affected per stockout day: 7.35 of 16


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 14

# Bar chart: % of stockout-days that first went empty at each hour. The opening-hour bar is
# picked out in a different colour, and the two headline stats are annotated directly on the chart.
fig, ax = plt.subplots(figsize=(7.6, 7.1), dpi=300)
colors = ["#2a78d6"] + ["#1baf7a"] * (len(hours) - 1)
ax.bar(hours, pct.values, color=colors, width=0.82, zorder=3)
ax.set_ylim(0, 19)
ax.annotate(f"already empty\nat opening ({already_open_pct:.1f}%)",
            xy=(hours[0], pct.values[0]), xytext=(hours[0] + 3.0, pct.values[0] + 2.6),
            fontsize=13, fontweight="bold", color="#0b0b0b", ha="left",
            arrowprops=dict(arrowstyle="-|>", lw=1.4, color="#0b0b0b"))
ax.text(0.97, 0.97, f"{never_recovers_pct:.1f}% of stockouts, once\nstarted, never restock\nbefore closing",
        transform=ax.transAxes, ha="right", va="top", fontsize=13, color="#2f5d56",
        bbox=dict(boxstyle="round,pad=0.5", fc="#1baf7a1f", ec="#2f5d56", lw=1.0))
ax.set_xticks(hours[::3])
ax.set_xticklabels([f"{h:02d}:00" for h in hours[::3]], fontsize=13)
ax.set_xlabel("hour the shelf first goes empty", fontsize=13.5, color="#52514e")
ax.set_ylabel("share of stockout-days (%)", fontsize=13.5, color="#52514e")
ax.tick_params(axis="y", labelsize=12, colors="#52514e")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=1, zorder=0)
ax.set_axisbelow(True)
ax.set_title("When stockouts first begin", fontsize=18, pad=20, loc="left")
fig.tight_layout()
fig.savefig(PLOTS / "stockout_timing.png", dpi=300, bbox_inches="tight", facecolor="white")
shutil.copy(PLOTS / "stockout_timing.png", POSTER_IMAGES / "stockout_timing.png")
plt.close(fig)
print("saved outputs/plots/stockout_timing.png (+ copied to poster/images/)")


saved outputs/plots/stockout_timing.png (+ copied to poster/images/)


In [ ]:
# Breakdown: does a stockout already under way AT OPENING behave differently from one that starts
# later - does it tend to stock back the same day, and how many separate empty spells does a day
# have? (reuses out_hours / first_hour / recovers_same_day / lo / hi from the cell above)
def _n_runs(hrs):
    """Separate empty-restocked-empty-again stretches in one day - almost always exactly 1."""
    hrs = sorted(hrs)
    runs = 1
    for a, b in zip(hrs, hrs[1:]):
        if b != a + 1:
            runs += 1
    return runs

runs = out_hours.apply(_n_runs)
opening = first_hour == lo
cohorts = {"already empty at opening (06:00)": opening,
           "goes empty later (07:00-21:00)": ~opening,
           "ALL stockout days": pd.Series(True, index=out_hours.index)}

rows = []
for label, mask in cohorts.items():
    n = int(mask.sum())
    rows.append(dict(
        cohort=label, n_stockout_days=n,
        share_of_stockout_days_pct=round(100 * n / len(out_hours), 1),
        recovers_before_close_pct=round(100 * float(recovers_same_day[mask].mean()), 1),
        stays_out_through_close_pct=round(100 * float((~recovers_same_day[mask]).mean()), 1),
        single_unbroken_run_pct=round(100 * float((runs[mask] == 1).mean()), 1),
    ))
recovery_by_start = pd.DataFrame(rows)
recovery_by_start.to_csv(REPORTS / "stockout_recovery_by_start.csv", index=False)
print("does an outage stock back the same day it starts, and how often?")
print(recovery_by_start.to_string(index=False))


## 4. Stage-wise results

Every table below is computed here from the saved artifacts (the recovery model board, the
four validation forecast parquets, the conformal reports), not transcribed from anywhere else.
If a number upstream changes, these change with it.

### Recovery: validated on held-out full-shelf days

In [4]:
ARMS = [(family, target) for family in ("tft", "xgb") for target in ("recovered", "raw")]

# Read the winning recovery model's own row back out of notebook 01's saved comparison table,
# plus the correction params and leakage verdicts - none of this is recomputed.
rec = pd.read_csv(config.RECOVERY_COMPARISON, index_col=0)
params, leak = (json.loads(config.RECOVERY_PARAMS.read_text()),
                json.loads(config.LEAKAGE_CHECKS.read_text()))
chosen, control = params["model"], "series_hour_mean"
recovery_tbl = pd.DataFrame([
    ("chosen Stage-1 model", chosen),
    ("day WAPE on held-out full-shelf days", f"{rec.loc[chosen, 'wape']:.4f}"),
    (f"no-model control ({control})", f"{rec.loc[control, 'wape']:.4f}"),
    ("better than the control by", f"{100 * (1 - rec.loc[chosen, 'wape'] / rec.loc[control, 'wape']):.1f}%"),
    ("aggregation bias before correction", f"{100 * rec.loc[chosen, 'wpe_uncorrected']:+.1f}%"),
    ("one fitted correction multiplier", f"x{rec.loc[chosen, 'bias_correction']:.4f}"),
    ("day WPE after correction", f"{rec.loc[chosen, 'wpe']:+.4f}"),
    ("leakage checks passed", f"{sum(leak.values())}/{len(leak)}"),
], columns=["what", "value"])
recovery_tbl.to_csv(REPORTS / "conclusion_recovery.csv")
print("DID RECOVERY WORK?  scored where recorded sales ARE demand, so it could have failed")
print("   source: reports/recovery_model_comparison.csv + recovery_params.json + leakage_checks.json")
print(recovery_tbl.to_string(index=False))

DID RECOVERY WORK?  scored where recorded sales ARE demand, so it could have failed
   source: reports/recovery_model_comparison.csv + recovery_params.json + leakage_checks.json
                                what            value
                chosen Stage-1 model lightgbm_tweedie
day WAPE on held-out full-shelf days           0.2966
 no-model control (series_hour_mean)           0.3684
          better than the control by            19.5%
  aggregation bias before correction           +13.2%
    one fitted correction multiplier          x0.9001
            day WPE after correction          +0.0186
               leakage checks passed              5/5


### Recovery, split by how often a series sells out: two model families, same answer

In [5]:
# Read straight back the per-band recovery table notebooks 02/03 already computed and saved.
by_band = pd.read_csv(REPORTS / "recovery_by_censoring_bucket.csv", index_col=0)
print(by_band.to_string(index=False))


   days  recorded_mean  recovered_mean  uplift_%  uplift_%_flat_fill  model_vs_flat_%
71589.0           1.41            1.52      7.98               14.75            54.10
83405.0           1.16            1.49     28.75               42.10            68.29
48999.0           0.83            1.40     69.52               99.31            70.00
32587.0           0.21            1.17    445.84              556.23            80.15


### The same split, on the live forecasts: does it change the forecast, not just Stage-1's own validation?

The table above is Stage-1's own validation (recovered vs. a no-model control, on days recorded
sales are known). This is a different question: does training the **forecaster** on recovered
demand instead of raw sales change its accuracy, and does the improvement concentrate in the same
chronic-stockout band, in **both** model families? Computed from the four validation forecast
parquets.

In [6]:
# Load all four arms' validation forecasts fresh, rather than relying on anything left over
# in the kernel from an earlier notebook.
val_fc = {f"{f}_{t}": pd.read_parquet(config.forecast_parquet("validation", t, f)) for f, t in ARMS}
band = censoring_bucket(daily)          # each series' share of TRAINING days that sold out

# For each model family, compare raw vs recovered accuracy per censoring band.
cols = {}
for fam in ("tft", "xgb"):
    t = pd.concat({tag: scores_by_bucket(val_fc[f"{fam}_{tag}"], band) for tag in ("raw", "recovered")},
                  axis=1)
    cols["n_scored"] = t[("raw", "n_scored")].astype(int)
    cols[f"{fam} WAPE change %"] = ((t[("recovered", "WAPE")] / t[("raw", "WAPE")] - 1) * 100).round(2)
    cols[f"{fam} WPE raw"] = t[("raw", "WPE")].round(3)
    cols[f"{fam} WPE recovered"] = t[("recovered", "WPE")].round(3)
band_tbl = pd.DataFrame(cols)
band_tbl.to_csv(REPORTS / "conclusion_by_band.csv")
print("DOES RECOVERY CHANGE THE FORECAST?  negative WAPE change = recovery is MORE accurate")
print("   computed live from the four validation forecast parquets, split by censoring band")
print(band_tbl.to_string())

DOES RECOVERY CHANGE THE FORECAST?  negative WAPE change = recovery is MORE accurate
   computed live from the four validation forecast parquets, split by censoring band
               n_scored  tft WAPE change %  tft WPE raw  tft WPE recovered  xgb WAPE change %  xgb WPE raw  xgb WPE recovered
<25% censored      3955              -0.57       -0.030              0.071              -1.45        0.020              0.118
25-50%            30345               3.16       -0.044              0.105               2.85        0.004              0.112
50-75%            12319               0.23       -0.067              0.084               1.23       -0.017              0.111
>=75%              1375             -25.24       -0.202             -0.003             -22.91       -0.205              0.001
ALL               47994              -0.06       -0.066              0.085               0.03       -0.022              0.100


### Does the same picture hold on the sealed test week?

The validation table above is a mixed bag: recovery clearly helps only the chronic-stockout band
and costs a little everywhere else, which is why a validation-only view can make recovery look
marginal. Repeating the identical by-band check on the test-week forecast (same `censoring_bucket`
labels, same `scores_by_bucket` scoring, nothing re-fit) is the honest way to see whether that
holds once real, previously-unseen demand is behind it.

In [7]:
test_fc = {f"{f}_{t}": pd.read_parquet(config.forecast_parquet("test", t, f)) for f, t in ARMS}
band_test = censoring_bucket(daily)   # same series -> band mapping, still fit on TRAINING days only

cols = {}
for fam in ("tft", "xgb"):
    t = pd.concat({tag: scores_by_bucket(test_fc[f"{fam}_{tag}"], band_test) for tag in ("raw", "recovered")},
                  axis=1)
    cols["n_scored"] = t[("raw", "n_scored")].astype(int)
    cols[f"{fam} WAPE change %"] = ((t[("recovered", "WAPE")] / t[("raw", "WAPE")] - 1) * 100).round(2)
    cols[f"{fam} WPE raw"] = t[("raw", "WPE")].round(3)
    cols[f"{fam} WPE recovered"] = t[("recovered", "WPE")].round(3)
band_tbl_test = pd.DataFrame(cols)
band_tbl_test.to_csv(REPORTS / "conclusion_by_band_test.csv")
print("SAME CHECK, SEALED TEST WEEK  -  negative WAPE change = recovery is MORE accurate")
print("   computed live from the four test forecast parquets, split by censoring band")
print(band_tbl_test.to_string())
print()
print("Contrast with validation: there, WPE raw is negative (under-forecast) in every band except")
print("chronic, where it swings to the over-forecast side once recovered - a mixed picture. Here,")
print("WPE raw stays negative in EVERY band, chronic included, and WAPE change is negative (i.e.")
print("recovery helps) in every band too - a one-sided picture, not a mixed one. That is the")
print("validation/test gap: it is not that recovery works differently on the test week, it is that")
print("the test week's higher realised demand (see Recovery vs. Model Choice) makes raw's")
print("under-forecast bias costly everywhere, not just in the chronic band.")

SAME CHECK, SEALED TEST WEEK  -  negative WAPE change = recovery is MORE accurate
   computed live from the four test forecast parquets, split by censoring band
               n_scored  tft WAPE change %  tft WPE raw  tft WPE recovered  xgb WAPE change %  xgb WPE raw  xgb WPE recovered
<25% censored      1822              -5.42       -0.235             -0.186              -0.89       -0.152             -0.065
25-50%            14380             -11.05       -0.246             -0.136              -3.73       -0.184             -0.086
50-75%             6086             -19.89       -0.296             -0.169              -9.60       -0.234             -0.117
>=75%              1114             -45.46       -0.411             -0.137             -26.22       -0.452             -0.216
ALL               23402             -19.29       -0.288             -0.151              -9.43       -0.239             -0.115

Contrast with validation: there, WPE raw is negative (under-forecast) in every ban

### Why would a higher-bias-but-larger week favour recovery more?

One more piece completes the picture: was the test week's demand actually higher than what the
models were fitted on, or is the by-band table above just a coincidence of one particular week?
Checked on clean (never-stocked-out) product-days only, so this is realised demand, not a
forecast.

In [8]:
clean = daily[daily["stock_hour6_22_cnt"] == 0]
demand_shift = (clean.groupby("period")["sale_amount"].mean()
                .reindex(["training", "validation", "calibration", "test"])
                .rename("avg_sale_amount_per_clean_day").reset_index())
demand_shift.to_csv(REPORTS / "conclusion_demand_shift.csv", index=False)
print("AVG sale_amount PER CLEAN PRODUCT-DAY, BY PERIOD")
print(demand_shift.to_string(index=False))
lift = 100 * (demand_shift.set_index("period").loc["test", "avg_sale_amount_per_clean_day"]
              / demand_shift.set_index("period").loc["training", "avg_sale_amount_per_clean_day"] - 1)
print()
print(f"test week vs. training window: {lift:+.1f}%")
print("Rises monotonically training -> validation -> calibration -> test, so this is a real drift,")
print("not a one-window fluke - and it is why raw's under-forecast bias, tuned on the training")
print("window's lower demand, gets more expensive by the time the test week arrives.")

AVG sale_amount PER CLEAN PRODUCT-DAY, BY PERIOD
     period  avg_sale_amount_per_clean_day
   training                       0.922441
 validation                       1.027558
calibration                       1.125698
       test                       1.250509

test week vs. training window: +35.6%
Rises monotonically training -> validation -> calibration -> test, so this is a real drift,
not a one-window fluke - and it is why raw's under-forecast bias, tuned on the training
window's lower demand, gets more expensive by the time the test week arrives.


### Are the bands honest?

Read directly off the saved conformal results for all four arms, both windows. Nothing here is
recomputed, this just assembles what `conformal.run` already wrote to disk. Alongside the raw
coverage numbers: a day-block bootstrap 95% CI on coverage, and a `nominal_in_ci` verdict column.
`True` means the band's coverage is statistically indistinguishable from what it claims,
`False` means the miss is real.

In [7]:
# Read straight out of the conformal report JSONs, so this table cannot drift from what was fitted.
# `nominal_in_ci` is the verdict column: True = the band's coverage is statistically
# indistinguishable from what it claims; False = the miss is real, and the sign says which way.
BANDS = [(0.80, "wide80"), (0.95, "wide95")]

rows = []
for family, target in ARMS:
    for nominal, tag in BANDS:
        path = config.conformal_results(tag, family=family, target=target)
        if not path.exists():
            continue
        for window, d in json.loads(path.read_text())["periods"].items():
            ci = d["coverage_ci"]
            rows.append({"arm": f"{family}_{target}", "band": f"{nominal:.0%}", "window": window,
                         "claims": nominal, "before": d["uncorrected"]["coverage"],
                         "after": d["corrected"]["coverage"],
                         "ci_low": ci["ci_low"], "ci_high": ci["ci_high"],
                         "offset": d["offset"], "drift": d["drift_inflation"],
                         "nominal_in_ci": bool(ci["ci_low"] <= nominal <= ci["ci_high"])})

calibration_tbl = (pd.DataFrame(rows)
                   .sort_values(["window", "band", "arm"], ascending=[False, True, True])
                   .reset_index(drop=True))
print("ARE THE BANDS HONEST?  does an 80% band contain the truth 80% of the time?")
print("   source: reports/conformal_results_<family>_<target>_<band>.json")
print(calibration_tbl.to_string(index=False))

n_pass = calibration_tbl.nominal_in_ci.sum()
print(f"\n   {n_pass} of {len(calibration_tbl)} (arm x band x window) combinations land statistically")
print("   on their nominal level. Note the pattern rather than the count:")
for w, g in calibration_tbl.groupby("window", sort=False):
    print(f"     {w:<11} raw coverage {g.before.min():.3f}-{g.before.max():.3f} -> "
          f"corrected {g.after.min():.3f}-{g.after.max():.3f}   "
          f"{g.nominal_in_ci.sum()}/{len(g)} contain nominal")
print("   validation OVERSHOOTS (bands slightly too wide); test UNDERSHOOTS on the 80% band for the")
print("   tree arms. One offset fitted at one distance in time cannot serve both distances at once.")

calibration_tbl.to_csv(REPORTS / "conclusion_calibration.csv", index=False)

ARE THE BANDS HONEST?  does an 80% band contain the truth 80% of the time?
   source: reports/conformal_results_<family>_<target>_<band>.json
          arm band     window  claims  before  after  ci_low  ci_high  offset  drift  nominal_in_ci
      tft_raw  80% validation    0.80  0.7494 0.8727  0.8579   0.8860  0.1216   0.00          False
tft_recovered  80% validation    0.80  0.7378 0.8255  0.8128   0.8376  0.0775   0.00          False
      xgb_raw  80% validation    0.80  0.7758 0.8584  0.8451   0.8710  0.0803   0.00          False
xgb_recovered  80% validation    0.80  0.7395 0.8455  0.8280   0.8612  0.0965   0.00          False
      tft_raw  95% validation    0.95  0.9215 0.9826  0.9767   0.9875  0.3140   0.00          False
tft_recovered  95% validation    0.95  0.9179 0.9702  0.9645   0.9753  0.1704   0.00          False
      xgb_raw  95% validation    0.95  0.9440 0.9774  0.9718   0.9819  0.1473   0.00          False
xgb_recovered  95% validation    0.95  0.9193 0.9703  0.96

### At matched 95% demand met: the waste_comparison chart

Every arm held to the same 95% demand-met target, so waste differences reflect forecast quality
rather than one arm simply ordering more (first table below). The deciding comparison is raw
against its own recovered twin at that identical availability (second table), the claim the
project stands on. A third view holds the cost ratio fixed instead (`q*=0.80`) and compares
every arm against the naive rule on both windows (third table).

In [8]:
# Built fresh here via orders.load_forecast rather than inherited from notebook 04's kernel -
# every arm that has a validation forecast, plus whichever also has a test forecast on disk
# (notebook 04 section 4 is what puts test forecasts there; this only checks what exists).
frames = {f"{f}_{t}": orders.load_forecast(period="validation", family=f, target=t) for f, t in ARMS}
test_frames = {}
for f, t in ARMS:
    try:
        test_frames[f"{f}_{t}"] = orders.load_forecast(period="test", family=f, target=t)
    except FileNotFoundError:
        pass
windows = {"validation": frames, "test": test_frames}

# Every arm held to the SAME availability, so waste is what forecast quality bought and not
# a side effect of one arm simply ordering more.
met = pd.concat([orders.at_demand_met(fr, 0.95, verbose=False).assign(window=w)
                 for w, fr in windows.items() if fr], ignore_index=True)
met = met[["window", "arm", "order_percentile", "waste_pct", "stockout_pct"]]
print("EVERY ARM HELD TO 95% OF DEMAND MET  -  what does each one waste to get there?")
print("    full-shelf days only: recorded sales ARE demand there, and those are the quiet days where")
print("    a recovered model over-orders, so this regime PENALISES recovery")
print(met.to_string(index=False))

# The deciding comparison: raw against its own recovered twin, same family, same availability.
# NOT waste_at_equal_service.csv - notebook 04 no longer writes that file; it only ever held
# whichever single validation-only period was last computed there. This is the two-window table
# behind Card 4's "waste at equal availability" row and the dashboard's waste_comparison.png,
# built below once both windows are open.
pairs = []
for w, g in met.groupby("window", sort=False):
    s = g.set_index("arm")
    for fam, nice in [("tft", "TFT"), ("xgb", "XGBoost")]:
        if not {f"{fam}_raw", f"{fam}_recovered"} <= set(s.index):
            continue
        r, c = s.loc[f"{fam}_raw"], s.loc[f"{fam}_recovered"]
        pairs.append({"window": w, "model": nice,
                      "raw waste %": r.waste_pct, "recovered waste %": c.waste_pct,
                      "waste saved (pts)": round(c.waste_pct - r.waste_pct, 1),
                      "raw orders at": f"q{r.order_percentile:.2f}",
                      "recovered orders at": f"q{c.order_percentile:.2f}"})
raw_vs_rec = pd.DataFrame(pairs)
print("\n\nRAW vs RECOVERED AT IDENTICAL AVAILABILITY  -  the claim the project stands on")
print(raw_vs_rec.to_string(index=False))
missing = [f"{f}_{t}" for f, t in ARMS if f"{f}_{t}" not in test_frames]
if missing:
    print(f"    no test forecast for: {', '.join(missing)} - those pairs are validation-only")

# The headline operating point against the status quo, every arm, both windows.
head = []
for w, fr in windows.items():
    for name, (df, qcols) in fr.items():
        k = orders.run(df, qcols, period=w, regime="observed", save=False, verbose=False)["kpi"]
        head.append({"window": w, "arm": name, "n_scored": k["n_scored"], **k["headline"]})
headline_tbl = pd.DataFrame(head).drop(columns=["c_u", "c_o"])
print("\n\nHEADLINE  q*=0.80 (a stockout assumed to cost 4x a bin) against the naive rule")
print("    positive cost_vs_naive_pct = CHEAPER than the naive rule")
print(headline_tbl.to_string(index=False))

for name, t in [("conclusion_raw_vs_recovered", raw_vs_rec), ("conclusion_headline", headline_tbl)]:
    t.to_csv(REPORTS / f"{name}.csv", index=False)

EVERY ARM HELD TO 95% OF DEMAND MET  -  what does each one waste to get there?
    full-shelf days only: recorded sales ARE demand there, and those are the quiet days where
    a recovered model over-orders, so this regime PENALISES recovery
    window           arm  order_percentile  waste_pct  stockout_pct
validation tft_recovered             0.696       41.4         15.55
validation xgb_recovered             0.679       43.2         14.66
validation       tft_raw             0.795       44.0         14.93
validation       xgb_raw             0.793       48.5         13.06
      test tft_recovered             0.920       51.9         10.54
      test       tft_raw             0.947       72.7          7.09
      test xgb_recovered             0.937       75.8          7.13
      test       xgb_raw             0.968       95.9          5.10


RAW vs RECOVERED AT IDENTICAL AVAILABILITY  -  the claim the project stands on
    window   model  raw waste %  recovered waste %  waste saved (



HEADLINE  q*=0.80 (a stockout assumed to cost 4x a bin) against the naive rule
    positive cost_vs_naive_pct = CHEAPER than the naive rule
    window           arm  n_scored  q_star  waste_vs_naive_pct  cost_vs_naive_pct  stockout_pct  demand_met_pct
validation tft_recovered     47994     0.8             -148.03              34.72          9.27           96.88
validation       tft_raw     47994     0.8             -104.00              37.26         14.56           95.12
validation xgb_recovered     47994     0.8             -173.52              30.68          7.74           97.24
validation       xgb_raw     47994     0.8             -125.67              32.76         12.58           95.16
      test tft_recovered     23402     0.8              -63.40              27.97         22.64           90.19
      test       tft_raw     23402     0.8              -39.41               9.22         29.23           84.55
      test xgb_recovered     23402     0.8             -117.14            

The poster's Card 4 shows this same table directly (plain numbers, no chart) - a grouped bar chart of these same figures looked weak and was dropped in favour of the table.

In [9]:
# The two-window raw-vs-recovered table was already written to disk by the demand-met comparison
# above - read it straight back rather than recomputing. A grouped-bar version of this chart was
# tried and dropped (see the markdown above); the poster's Card 4 just shows these numbers directly.
raw_vs_rec = pd.read_csv(REPORTS / "conclusion_raw_vs_recovered.csv")
print("Card 4's table reads this directly - no chart needed:")
print(raw_vs_rec.to_string(index=False))

Card 4's table reads this directly - no chart needed:
    window   model  raw waste %  recovered waste %  waste saved (pts) raw orders at recovered orders at
validation     TFT         44.0               41.4               -2.6         q0.80               q0.70
validation XGBoost         48.5               43.2               -5.3         q0.79               q0.68
      test     TFT         72.7               51.9              -20.8         q0.95               q0.92
      test XGBoost         95.9               75.8              -20.1         q0.97               q0.94


## 5. Final ordering decision: the cost sweep

Validation window, recovered TFT arm, from `reports/cost_sweep.csv`. The sealed test week is scored
only at the single headline point (§4 above and the four-arm table in §6).

In [10]:
# Read back the cost sweep notebook 04 already computed and saved - nothing recomputed here.
sweep = pd.read_csv(REPORTS / "cost_sweep.csv")
print(sweep.to_string(index=False))
print()
print(f"cheaper than naive at all {len(sweep)} ratios tested, "
      f"{sweep.cost_vs_naive_pct.min():.1f}% to {sweep.cost_vs_naive_pct.max():.1f}%")

  c_u  c_o  q_star  waste_model  waste_naive  waste_vs_naive_pct  stockout_pct_model  stockout_pct_naive  demand_met_pct_model  demand_met_pct_naive  cost_model  cost_naive  cost_vs_naive_pct
 0.25  1.0  0.2000      2186.79     10792.39               79.74               77.47                41.7                 66.70                 79.91     6292.39    13269.74              52.58
 0.50  1.0  0.3333      4830.16     10792.39               55.24               59.53                41.7                 77.95                 79.91    10266.17    15747.08              34.81
 1.00  1.0  0.5000     10186.52     10792.39                5.61               36.54                41.7                 87.86                 79.91    16172.70    20701.77              21.88
 2.00  1.0  0.6667     18703.70     10792.39              -73.30               17.93                41.7                 94.28                 79.91    24343.60    30611.16              20.47
 3.00  1.0  0.7500     23651.35     1079

In [11]:
# Same cost-sweep line as before, but now overlaid with the identical sweep run on the sealed
# test week (tft_recovered only) - so the chart itself shows whether the saving holds out of
# sample, not just the single number quoted in the markdown above.
test_sweep = orders.run(*orders.load_forecast(period="test", family="tft", target="recovered"),
                        period="test", regime="observed", save=False, verbose=False)["sweep"]

# Two lines, one per window, both indexed the same way: % cheaper than the naive rule at each
# assumed cost ratio. A flat line at 0% would mean 'no better than current practice'.
fig, ax = plt.subplots(figsize=(9.8, 6.0), dpi=300)
x = range(len(sweep))
ax.axhline(0, color="#898781", linewidth=2.2, linestyle=(0, (5, 3)), zorder=2,
           label="current practice: order what sold last week")
ax.plot(x, sweep["cost_vs_naive_pct"], marker="o", markersize=8, linewidth=2.6, color="#eb6834",
        zorder=3, label="recovered-demand model (checked on the practice weeks)")
ax.plot(x, test_sweep["cost_vs_naive_pct"], marker="D", markersize=8, linewidth=2.6, color="#2f5d56",
        zorder=4, label="same model (checked on the sealed-away test week)")
for i, v in zip(x, sweep["cost_vs_naive_pct"]):
    ax.annotate(f"{v:.0f}%", (i, v), textcoords="offset points", xytext=(0, 12),
                ha="center", fontsize=11.5, fontweight="bold", color="#eb6834")
for i, v in zip(x, test_sweep["cost_vs_naive_pct"]):
    ax.annotate(f"{v:.0f}%", (i, v), textcoords="offset points", xytext=(0, -18),
                ha="center", fontsize=11.5, fontweight="bold", color="#2f5d56")
ax.set_xticks(list(x))
ax.set_xticklabels([f"{v:g}×" for v in sweep["c_u"]], fontsize=13)
ax.set_xlabel("assumed cost ratio  (how much worse an empty shelf is than a wasted item)",
              fontsize=12.5, color="#52514e")
ax.set_ylabel("% cheaper than current practice\n(0% = same cost)", fontsize=13, color="#52514e")
ax.tick_params(axis="y", labelsize=12, colors="#52514e")
ax.set_ylim(-15, 80)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.spines["left"].set_color("#c3c2b7")
ax.spines["bottom"].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=1, zorder=0)
ax.set_axisbelow(True)
ax.set_title("Cheaper than current practice at every cost assumption, on both weeks",
             fontsize=16.5, pad=14, loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=1, frameon=False, fontsize=12)
fig.tight_layout()
fig.savefig(PLOTS / "cost_sweep.png", dpi=300, bbox_inches="tight", facecolor="white")
shutil.copy(PLOTS / "cost_sweep.png", POSTER_IMAGES / "cost_sweep.png")
plt.close(fig)
print("saved outputs/plots/cost_sweep.png (+ copied to poster/images/)")
print()
print("-- practice weeks --")
print(sweep[["c_u", "cost_vs_naive_pct"]].to_string(index=False))
print("-- sealed test week --")
print(test_sweep[["c_u", "cost_vs_naive_pct"]].to_string(index=False))

saved outputs/plots/cost_sweep.png (+ copied to poster/images/)

-- practice weeks --
  c_u  cost_vs_naive_pct
 0.25              52.58
 0.50              34.81
 1.00              21.88
 2.00              20.47
 3.00              27.37
 4.00              34.72
 6.00              46.46
 9.00              57.76
19.00              72.56
-- sealed test week --
  c_u  cost_vs_naive_pct
 0.25              42.82
 0.50              22.49
 1.00              10.64
 2.00              13.93
 3.00              21.30
 4.00              27.97
 6.00              37.82
 9.00              46.87
19.00              66.90


### 5b. How much should the 34.7%/28.0% wobble?

Both numbers are averages over a handful of calendar days (14 validation, 7 test) - not
independent rows, since every product-day sharing a date moves together. Day-block
bootstrap, same trick as `conformal.coverage_ci`, applied to the per-day cost outcome
instead of the per-row coverage hit.

In [12]:
def cost_ci(per_day: pd.DataFrame, n_boot: int = 2000, seed: int = 0) -> dict:
    """Day-block bootstrap CI on cost_vs_naive_pct - resample whole calendar DATES, not
    rows, for the same reason conformal.coverage_ci does: every product-day sharing a
    date is not an independent draw."""
    g = per_day.groupby("dt")[["model_cost", "naive_cost"]].sum()
    m, n = g["model_cost"].to_numpy(), g["naive_cost"].to_numpy()
    n_days = len(g)
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        pick = rng.integers(0, n_days, n_days)
        draws.append(100 * (1 - m[pick].sum() / n[pick].sum()))
    lo, hi = np.percentile(draws, [2.5, 97.5])
    point = 100 * (1 - m.sum() / n.sum())
    return dict(point=round(float(point), 2), ci_low=round(float(lo), 2),
                ci_high=round(float(hi), 2), n_days=int(n_days))


# Run the bootstrap for both windows, on the recovered TFT at the headline ratio - the same
# operating point the four-arm headline table already reports a single number for. The assert
# below checks the bootstrap's own point estimate reproduces that number, so a bug in the
# day-grouping couldn't silently drift from what orders.run actually computed.
ci_rows = []
for w in ("validation", "test"):
    df_w, qcols_w = orders.load_forecast(period=w, family="tft", target="recovered")
    out_w = orders.run(df_w, qcols_w, period=w, regime="observed", c_u=orders.HEADLINE_RATIO,
                        save=False, verbose=False)
    ci = cost_ci(out_w["per_day"])
    assert abs(ci["point"] - out_w["kpi"]["headline"]["cost_vs_naive_pct"]) < 0.05
    ci_rows.append(dict(window=w, **ci))
    print(f"[{w}] cost_vs_naive_pct = {ci['point']}%  (95% CI {ci['ci_low']}-{ci['ci_high']}%, "
          f"{ci['n_days']} calendar days)")

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(REPORTS / "cost_savings_ci.csv", index=False)
print()
print(ci_df.to_string(index=False))

[validation] cost_vs_naive_pct = 34.72%  (95% CI 29.98-39.37%, 14 calendar days)


[test] cost_vs_naive_pct = 27.97%  (95% CI 25.42-31.36%, 7 calendar days)

    window  point  ci_low  ci_high  n_days
validation  34.72   29.98    39.37      14
      test  27.97   25.42    31.36       7


## 6. Choosing an operating point

Validation window, full-shelf days, recovered TFT arm. Computed below rather than read back,
since the strict-dominance window it finds is worth showing the search for, not just the result.

In [13]:
# EVERY ROW HERE IS SELF-CONSISTENT. `q*` and `c_u/c_o` are one number written two ways
# (q* = c_u/(c_u+c_o), so with c_o fixed at 1 the ratio is exactly q*/(1-q*)). Each row therefore
# reads "IF you believe a stockout costs this much, order here, and these are the outcomes."
# The ratio is NOT held fixed down the table - it varies with q*, by definition. The next
# subsection does the opposite, and says so there.
#
# Neither q* nor the ratio means anything to a shopkeeper, so both are mapped onto outcomes a shop
# feels, and compared against the NAIVE RULE - the one reference point needing no cost assumption at all.
model_df, model_qcols = orders.load_forecast(period="validation", family="tft", target="recovered")
demand, mask = orders.realised_demand(model_df, "observed")
naive_order = orders.naive_orders(model_df)[mask]
total = demand.sum()

def outcome(q_star):
    """(waste units, % of demand met, % of product-days that ran out) at one order percentile."""
    o = orders.order_quantity(model_df, q_star, model_qcols)[mask]
    return (float(np.maximum(o - demand, 0).sum()),
            100 * float(1 - np.maximum(demand - o, 0).sum() / total),
            100 * float((demand > o).mean()))

naive_waste = float(np.maximum(naive_order - demand, 0).sum())
naive_met = 100 * float(1 - np.maximum(demand - naive_order, 0).sum() / total)
naive_out = 100 * float((demand > naive_order).mean())
print(f"NAIVE RULE: waste {naive_waste:,.0f} units | demand met {naive_met:.1f}% | "
      f"ran out on {naive_out:.1f}% of product-days")

# Three boundaries against naive, each ONE-SIDED and reported separately. No waste ceiling is
# invented: a tolerable waste level is a business constraint nobody supplied, and inventing one is
# exactly what the cost sweep exists to avoid. An earlier version of this cell collapsed these into a
# single "viable" flag, which passed a policy wasting 4x the naive rule as fine. Three columns, no verdict.
grid = np.arange(0.30, 0.99, 0.001)
stats = np.array([outcome(q) for q in grid])
q_waste_ok = float(grid[max(int(np.argmax(stats[:, 0] > naive_waste)) - 1, 0)])   # waste <= naive
q_met_ok = float(grid[int(np.argmax(stats[:, 1] >= naive_met))])                  # demand met >= naive
q_out_ok = float(grid[int(np.argmax(stats[:, 2] <= naive_out))])                  # ran out <= naive
lo, hi = max(q_met_ok, q_out_ok), q_waste_ok
print(f"\nBOUNDARIES vs naive:  demand met >= naive from q{q_met_ok:.3f}  |  "
      f"ran out <= naive from q{q_out_ok:.3f}  |  waste <= naive up to q{q_waste_ok:.3f}")
print(f"\nSTRICT-DOMINANCE WINDOW   q{lo:.3f} to q{hi:.3f}")
print("  Inside it the model beats the naive rule on ALL THREE at once - less waste, more demand met,")
print("  fewer empty days - so it needs NO cost assumption to justify. Outside it you are making a")
print("  trade, and a trade requires a ratio you can defend.")
for q_star, label in ((lo, "window floor"), (hi, "window ceiling = waste-neutral")):
    w, m, k = outcome(q_star)
    print(f"    q{q_star:.3f} ({label:31s}) waste {w:8,.0f} vs {naive_waste:,.0f} | "
          f"met {m:.1f}% vs {naive_met:.1f}% | ran out {k:.1f}% vs {naive_out:.1f}%")

# The reader-facing map. Each comparison is its own column, stated as a direction, so nothing hides
# inside a summary verdict.
rows = []
for q_star in sorted({0.33, 0.40, round(lo, 3), 0.50, round(hi, 3), 0.60, 0.6667, 0.75, 0.80, 0.90, 0.95}):
    w, m, k = outcome(q_star)
    rows.append({"order at": f"q{q_star:.3f}", "implied c_u/c_o": round(q_star / (1 - q_star), 2),
                 "ran out %": round(k, 1), "empty days": "better" if k <= naive_out else "WORSE",
                 "demand met %": round(m, 1), "availability": "better" if m >= naive_met else "WORSE",
                 "waste % of demand": round(100 * w / total, 1),
                 "waste vs naive %": round(100 * (1 - w / naive_waste), 0),
                 "waste": "better" if w <= naive_waste else "WORSE"})
qmap = pd.DataFrame(rows)
print("\n\nTHE ORDER PERCENTILE, TRANSLATED   validation, full-shelf days, recovered TFT")
print("    each row is self-consistent - the ratio shown is the one that q* implies, not a fixed one")
print(qmap.to_string(index=False))
print("\n    Read the three verdict columns together. Above the window the model buys availability")
print("    with waste. That is a legitimate choice - but it is a CHOICE, and it only becomes")
print("    defensible once the cost ratio behind it is.")

qmap.to_csv(REPORTS / "conclusion_quantile_map.csv", index=False)


TODAY (the naive rule): waste 10,792 units | demand met 79.9% | ran out on 41.7% of product-days



BOUNDARIES vs today:  demand met >= today from q0.361  |  ran out <= today from q0.461  |  waste <= today up to q0.513

STRICT-DOMINANCE WINDOW   q0.461 to q0.513
  Inside it the model beats the status quo on ALL THREE at once - less waste, more demand met,
  fewer empty days - so it needs NO cost assumption to justify. Outside it you are making a
  trade, and a trade requires a ratio you can defend.
    q0.461 (window floor                   ) waste    8,732 vs 10,792 | met 86.0% vs 79.9% | ran out 41.7% vs 41.7%
    q0.513 (window ceiling = waste-neutral ) waste   10,768 vs 10,792 | met 88.5% vs 79.9% | ran out 34.9% vs 41.7%


THE ORDER PERCENTILE, TRANSLATED   validation, full-shelf days, recovered TFT
    each row is self-consistent - the ratio shown is the one that q* implies, not a fixed one
order at  implied c_u/c_o  ran out % empty days  demand met % availability  waste % of demand  waste vs today %  waste
  q0.330             0.49       60.0      WORSE          77.7        W

## 6b. New check: do the validation-chosen quantiles survive the test week?

The three anchors above (`waste-focused`, `balanced`, `stockout-focused`) are quantiles picked on
**validation**. This section takes those exact, already-fixed quantiles and applies them, unchanged,
to the **sealed test week**, for both model families, to see whether the choice made on
validation actually holds up out of sample. This had never been run before; it exists because the
poster claims the three anchors as safe choices, and that claim should survive contact with the
test week the same way every other number here is required to.

In [14]:
ANCHOR_Q = {"waste_focused": 0.461, "balanced": 0.513, "stockout_focused": 0.90}

# (stockout %, demand met %, waste %) for one order array against actual demand - defined here
# since this is the first section in the notebook that needs it.
def score(order_arr, demand):
    sim = orders.simulate(order_arr, demand)
    return (100 * sim.stockout.mean(), 100 * (1 - sim.shortfall.sum() / demand.sum()),
            100 * sim.waste.sum() / demand.sum())
# Apply the three validation-chosen anchors, completely unchanged, to both windows and both
# model families - see the markdown above for why this out-of-sample check exists.
rows = []
for family in ("tft", "xgb"):
    for period in ("validation", "test"):
        df, qcols = orders.load_forecast(period=period, family=family, target="recovered")
        demand, mask = orders.realised_demand(df, regime="observed")
        naive_order = orders.naive_orders(df)

        so, dm, w = score(naive_order[mask], demand)
        rows.append(dict(family=family, period=period, anchor="naive_today",
                         stockout_pct=round(so, 1), demand_met_pct=round(dm, 1), waste_pct=round(w, 1)))
        for name, q in ANCHOR_Q.items():
            order_arr = orders.order_quantity(df, q, qcols)[mask]
            so, dm, w = score(order_arr, demand)
            rows.append(dict(family=family, period=period, anchor=name,
                             stockout_pct=round(so, 1), demand_met_pct=round(dm, 1), waste_pct=round(w, 1)))

transfer = pd.DataFrame(rows)
transfer.to_csv(REPORTS / "quantile_transfer_to_test.csv", index=False)
print(transfer.to_string(index=False))
print()
print("Reading: waste-focused and balanced were chosen on validation because they matched or beat")
print("naive on stockouts there. On the test week, both arms show BOTH anchors doing WORSE than")
print("naive on stockouts and demand met (only cheaper on waste) - only stockout-focused (q0.90)")
print("stays unambiguously better than naive on the test week, in both model families.")


family     period           anchor  stockout_pct  demand_met_pct  waste_pct
   tft validation      naive_today          41.7            79.9       21.9
   tft validation    waste_focused          41.7            86.0       17.7
   tft validation         balanced          34.9            88.5       21.8
   tft validation stockout_focused           5.6            98.0       67.4
   tft       test      naive_today          42.8            80.5       18.6
   tft       test    waste_focused          60.6            73.0        7.8
   tft       test         balanced          53.9            76.3       10.1
   tft       test stockout_focused          16.0            92.9       39.5
   xgb validation      naive_today          41.7            79.9       21.9
   xgb validation    waste_focused          40.0            86.0       19.0
   xgb validation         balanced          32.9            88.8       23.6
   xgb validation stockout_focused           4.5            98.3       74.4
   xgb      

## 6c. One store, illustrated

The 100-store aggregate above is the evidence; this is what it looks like for one real store on
the sealed test week. `store_view.pick_store` finds the closest-to-median store in the subset by
censoring rate (ties broken by product count) - not a cherry-picked best case. Same newsvendor
machinery as everywhere else in this project, at a stockout costing twice as much as a wasted
item, the ratio the poster's one-store card quotes.

In [15]:
# Pick one representative store - closest to the median censoring rate in the subset, not
# cherry-picked - so this illustration can't be accused of showing the best case.
store_id = store_view.pick_store(store_view.store_stats(daily))
# A stockout costing twice as much as a wasted item - the ratio the poster's one-store card quotes.
ratio = 2.0
q_star = orders.critical_fractile(ratio, 1.0)

# Slice that one store's test-week forecast out of the full arm, and attach both order
# policies to it so the plot below can draw them side by side.
test_df, test_qcols = orders.load_forecast(period="test", family="tft", target="recovered")
store_slice = test_df[test_df["store_id"] == store_id].copy()
store_slice["model_order_quantity"] = orders.order_quantity(store_slice, q_star, test_qcols)
store_slice["naive_order_quantity"] = orders.naive_orders(store_slice)

# Run the same newsvendor simulation as every other arm in this notebook, just restricted to
# this one store's rows.
store_result = orders.run(forecast_df=store_slice, qcols=test_qcols, period="test",
                          regime="observed", c_u=ratio, c_o=1.0, save=False, verbose=False)
store_kpi = store_result["kpi"]["headline"]
store_waste_pct = (100 * store_result["per_day"]["model_waste"].sum()
                   / store_result["per_day"]["sale_amount"].sum())

print(f"store #{store_id} (closest to median censoring rate in the 100-store subset), sealed "
      f"test week, a stockout costing {ratio:g}x a wasted item")
print(f"  stockout days {store_kpi['stockout_pct']:.1f}% | demand met {store_kpi['demand_met_pct']:.1f}% | "
      f"cheaper than naive {store_kpi['cost_vs_naive_pct']:.1f}% | waste {store_waste_pct:.1f}%")

# Draw the store's actual week: demand against both order policies, and where each one over-
# or under-shoots it.
plots.plot_store_week(store_slice, store_id, save_path=PLOTS / "store_week_example.png")
shutil.copy(PLOTS / "store_week_example.png", POSTER_IMAGES / "store_week_example.png")
print("saved outputs/plots/store_week_example.png (+ copied to poster/images/)")

store #76 (closest to median censoring rate in the 100-store subset), sealed test week, a stockout costing 2x a wasted item
  stockout days 34.7% | demand met 90.5% | cheaper than naive 26.6% | waste 17.1%


saved outputs/plots/store_week_example.png (+ copied to poster/images/)


## 7. Why TFT and recovered data

### Forecast accuracy, scored on non-stockout (clean) rows

Computed here from the four validation forecast parquets, not read from
`forecast_vs_baselines.csv`. The latter is an older, TFT-only scorecard from before the
XGBoost arms were added, and would silently drop `xgb_recovered`/`xgb_raw` from the comparison
the poster's Card 7 actually makes.

In [16]:
# Load all four arms' validation forecasts fresh - deliberately not the older, TFT-only
# forecast_vs_baselines.csv (see the markdown above for why).
val_fc = {f"{f}_{t}": pd.read_parquet(config.forecast_parquet("validation", t, f)) for f, t in ARMS}
arms_acc = pd.DataFrame({n: {**quantile_scores(d), "pinball@0.8": pinball_by_quantile(d)["pinball@0.8"]}
                         for n, d in val_fc.items()}).T
accuracy = pd.concat([arms_acc,
                      pd.read_csv(config.BASELINE_SCORECARD, index_col=0)[["WAPE", "WPE"]]]).round(4)
accuracy.to_csv(REPORTS / "conclusion_accuracy.csv")
print("FORECAST ACCURACY  validation, non-stockout rows, against recorded sale_amount")
print(accuracy.to_string())
print()
print("pinball@0.8 is the only accuracy column that converts into money - it is the quantile the")
print("order is read at. Recovered beats raw there in BOTH families, while pooled WAPE ties.")
print()
print(f"TFT vs XGBoost, recovered: pinball {accuracy.loc['tft_recovered', 'pinball(avg)']:.4f} "
      f"vs {accuracy.loc['xgb_recovered', 'pinball(avg)']:.4f} "
      f"({100*(1 - accuracy.loc['tft_recovered','pinball(avg)']/accuracy.loc['xgb_recovered','pinball(avg)']):.1f}% better)")

FORECAST ACCURACY  validation, non-stockout rows, against recorded sale_amount
                    WAPE     WPE     MAE  pinball(avg)   CRPS~  pinball@0.8
tft_recovered     0.3284  0.0851  0.3358        0.1074  0.2091       0.1281
tft_raw           0.3286 -0.0656  0.3364        0.1089  0.2120       0.1300
xgb_recovered     0.3417  0.1005  0.3502        0.1116  0.2173       0.1330
xgb_raw           0.3416 -0.0218  0.3495        0.1118  0.2176       0.1362
seasonal_naive    0.4213  0.0187     NaN           NaN     NaN          NaN
xgboost_quantile  0.3423 -0.0187     NaN           NaN     NaN          NaN
sarima            0.5210 -0.2521     NaN           NaN     NaN          NaN

pinball@0.8 is the only accuracy column that converts into money - it is the quantile the
order is read at. Recovered beats raw there in BOTH families, while pooled WAPE ties.

TFT vs XGBoost, recovered: pinball 0.1074 vs 0.1116 (3.8% better)


### The payoff, quantified: lost sales recovered vs. waste added

At q50, on full-shelf days only (the regime that *penalises* recovery), for both model families.
`worth_it_above_ratio` = waste added ÷ lost sales recovered: how much worse an empty shelf has to be
than a bin before recovery pays. Below 1.0 means it pays even if the two cost exactly the same.

In [17]:
band = censoring_bucket(daily)

# Same lost-sales-vs-waste trade table as notebooks 02/03, run here for both model families
# back to back so they can be read one after another.
for family in ("tft", "xgb"):
    raw_df, raw_qcols = orders.load_forecast(period="validation", family=family, target="raw")
    rec_df, rec_qcols = orders.load_forecast(period="validation", family=family, target="recovered")
    trade = lost_sales_vs_waste({"raw": raw_df, "recovered": rec_df}, band)
    print(f"--- {family} ---")
    print(trade.to_string())
    print()


--- tft ---
               lost_sales_%_raw  waste_%_raw  lost_sales_%_recovered  waste_%_recovered  lost_sales_recovered_pts  waste_added_pts  worth_it_above_ratio
band                                                                                                                                                    
<25% censored              22.8         19.5                    17.6               24.4                       5.2              4.9                  0.94
25-50%                     19.8         15.4                    12.8               23.3                       7.0              7.9                  1.13
50-75%                     18.1         11.6                    10.8               19.0                       7.3              7.4                  1.01
>=75%                      22.7          2.8                     9.8                9.1                      12.9              6.3                  0.49

--- xgb ---
               lost_sales_%_raw  waste_%_raw  lost_sales_

### On the sealed test week, recovery decides it

All 4 versions against current practice, headline ratio only, sealed test week - the window that matters. On the practice weeks this gap barely shows (31-37% either way); it only opens up here.

The poster's own card shows this as a plain table (Table 2, same call as Card 4: a grouped-bar version looked weak next to the numbers). The chart below is kept here for the record.

In [18]:
# Read back the four-arm headline table (see the markdown above) and pull out just the test-week
# rows - this is the comparison that only opens up once the sealed week is actually scored.
headline = pd.read_csv(REPORTS / "conclusion_headline.csv")
sub = headline.query("window == 'test'").set_index("arm")
families = [("tft", "TFT"), ("xgb", "XGBoost")]
x = [0, 0.9]
raw_vals = [sub.loc[f"{f}_raw", "cost_vs_naive_pct"] for f, _ in families]
rec_vals = [sub.loc[f"{f}_recovered", "cost_vs_naive_pct"] for f, _ in families]

# Grouped bar chart: raw vs. recovered cost-vs-naive, side by side per model family, test week only.
fig, ax = plt.subplots(figsize=(7.6, 5.4), dpi=300)
width = 0.16
ax.axhline(0, color="#898781", linewidth=1.8, linestyle=(0, (5, 3)), zorder=2)
b1 = ax.bar([i - width / 2 - 0.015 for i in x], raw_vals, width, color="#2a78d6",
            label="trained on raw sales", zorder=3)
b2 = ax.bar([i + width / 2 + 0.015 for i in x], rec_vals, width, color="#eb6834",
            label="trained on recovered demand", zorder=3)
for bars, vals in ((b1, raw_vals), (b2, rec_vals)):
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.8, f"{v:.1f}%", ha="center", va="bottom",
                fontsize=13, color="#0b0b0b")
ax.set_xticks(x)
ax.set_xticklabels([nice for _, nice in families], fontsize=15)
ax.set_xlim(-0.55, 1.45)
ax.set_ylabel("% cheaper than current practice", fontsize=13, color="#52514e")
ax.tick_params(axis="y", labelsize=12, colors="#52514e")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.spines["left"].set_color("#c3c2b7")
ax.spines["bottom"].set_color("#c3c2b7")
ax.yaxis.grid(True, color="#e1e0d9", linewidth=1, zorder=0)
ax.set_axisbelow(True)
ax.set_ylim(0, max(raw_vals + rec_vals) + 8)
ax.set_title("On the sealed test week, recovery is what wins", fontsize=16, pad=14, loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=1, frameon=False, fontsize=12.5)
fig.tight_layout()
fig.savefig(PLOTS / "naive_comparison_all_arms.png", dpi=300, bbox_inches="tight", facecolor="white")
shutil.copy(PLOTS / "naive_comparison_all_arms.png", POSTER_IMAGES / "naive_comparison_all_arms.png")
plt.close(fig)
print(headline[["window", "arm", "cost_vs_naive_pct"]].to_string(index=False))
print()
print("saved outputs/plots/naive_comparison_all_arms.png (+ copied to poster/images/)")

    window           arm  cost_vs_naive_pct
validation tft_recovered              34.72
validation       tft_raw              37.26
validation xgb_recovered              30.68
validation       xgb_raw              32.76
      test tft_recovered              27.97
      test       tft_raw               9.22
      test xgb_recovered              13.40
      test       xgb_raw               0.84

saved outputs/plots/naive_comparison_all_arms.png (+ copied to poster/images/)


### Decomposing the win: recovery's effect vs. model choice's effect

Two 2x2 comparisons on the same headline table - hold architecture fixed and vary the
training target (recovery's effect), then hold the target fixed and vary the architecture
(model choice's effect). Card 8's numbers table.

In [19]:
# Reshape the same headline table into a lookup by (window, arm), so the two comparisons below
# can just difference the right cells straight out of it.
pivot = headline.set_index(["window", "arm"])["cost_vs_naive_pct"]

# Hold architecture fixed, vary the training target - isolates what recovery itself is worth.
recovery_effect = {
    (fam, w): pivot[(w, f"{fam}_recovered")] - pivot[(w, f"{fam}_raw")]
    for fam in ("tft", "xgb") for w in ("validation", "test")
}
# Hold the target fixed, vary architecture - isolates what picking TFT over XGBoost is worth.
model_effect = {
    (target, w): pivot[(w, f"tft_{target}")] - pivot[(w, f"xgb_{target}")]
    for target in ("raw", "recovered") for w in ("validation", "test")
}

print("Recovery's effect (recovered - raw), percentage points:")
for fam in ("tft", "xgb"):
    print(f"  {fam.upper():8s} validation {recovery_effect[(fam,'validation')]:+.1f}   "
          f"test {recovery_effect[(fam,'test')]:+.1f}")

print()
print("Model choice's effect (TFT - XGBoost), percentage points:")
for target in ("raw", "recovered"):
    print(f"  {target:10s} validation {model_effect[(target,'validation')]:+.1f}   "
          f"test {model_effect[(target,'test')]:+.1f}")

Recovery's effect (recovered - raw), percentage points:
  TFT      validation -2.5   test +18.8
  XGB      validation -2.1   test +12.6

Model choice's effect (TFT - XGBoost), percentage points:
  raw        validation +4.5   test +8.4
  recovered  validation +4.0   test +14.6
